# USSY TrendFoll 1 — Backtest Stop Anchor A/B
Membandingkan stop berbasis **Trigger T0** dengan **Filled/Open H+1**. ATR tetap memakai nilai T0.

In [ ]:
"""
USSY TrendFoll — Backtest A/B Stop Anchor (Google Colab)
========================================================

Membandingkan hanya satu keputusan strategi:

    A. trigger_t0 : stop = close Trigger T0 - 2 * ATR T0
    B. filled_h1  : stop = open/fill H+1   - 2 * ATR T0

ATR selalu memakai ATR T0 agar tidak memakai informasi setelah sinyal.
Entry selalu open H+1. Filter momentum+gap tidak digunakan karena backtest
sebelumnya tidak mendukung penambahannya.

Cara pakai:
1. Pastikan folder Google Drive dan nama file pada bagian CONFIG benar.
2. Upload/simpan hard_filter.py atau ussy_swing_hard_filter.py di folder itu.
3. Jalankan seluruh sel/script di Google Colab.
4. Unduh CSV ringkasan dan, bila perlu, kedua trade log.
"""

from pathlib import Path
import shutil
import sys

import numpy as np
import pandas as pd
from google.colab import drive, files


# ============================================================
# CONFIG
# ============================================================

DRIVE_FOLDER = "/content/drive/MyDrive/ussy-swing"
FEATURE_STORE_PATH = f"{DRIVE_FOLDER}/ussy_swing_feature_store.parquet"

ATR_STOP_MULTIPLIER = 2.0
MAX_HOLDING_DAYS = 45
VOLUME_PCT_THRESHOLD = 80
RISK_PCT = 0.01
INITIAL_CAPITAL = 100_000.0
MAX_CONCURRENT_POSITIONS = 20

VARIANTS = {
    "trigger_t0": "trigger",
    "filled_h1": "filled",
}


# ============================================================
# SETUP
# ============================================================

drive.mount("/content/drive")
sys.path.insert(0, DRIVE_FOLDER)

try:
    from hard_filter import compute_hard_filter
except ImportError:
    # Kompatibilitas dengan nama lama sebelum proyek dinamai TrendFoll.
    from ussy_swing_hard_filter import compute_hard_filter


def prepare_data(feature_store: pd.DataFrame):
    """Hitung hard filter dan lookup historis satu kali untuk kedua varian."""
    filtered = compute_hard_filter(feature_store)
    filtered = filtered.sort_values(["date", "symbol"]).reset_index(drop=True)

    dates = sorted(filtered["date"].unique())
    by_date = {d: g.set_index("symbol") for d, g in filtered.groupby("date")}

    symbol_row_by_date = {}
    symbol_index_by_date = {}
    symbol_prev_pivot = {}
    symbol_prev_close = {}

    for symbol, group in filtered.groupby("symbol"):
        group = group.sort_values("date").reset_index(drop=True).copy()
        group["prev_pivot_high"] = group["pivot_high"].shift(1)
        group["prev_close"] = group["close_raw"].shift(1)

        symbol_row_by_date[symbol] = group.set_index("date").to_dict("index")
        symbol_index_by_date[symbol] = {
            date: index for index, date in enumerate(group["date"])
        }
        symbol_prev_pivot[symbol] = (
            group.set_index("date")["prev_pivot_high"].to_dict()
        )
        symbol_prev_close[symbol] = group.set_index("date")["prev_close"].to_dict()

    return {
        "dates": dates,
        "by_date": by_date,
        "symbol_row_by_date": symbol_row_by_date,
        "symbol_index_by_date": symbol_index_by_date,
        "symbol_prev_pivot": symbol_prev_pivot,
        "symbol_prev_close": symbol_prev_close,
    }


def backtest(prepared, stop_anchor):
    dates = prepared["dates"]
    by_date = prepared["by_date"]
    symbol_row_by_date = prepared["symbol_row_by_date"]
    symbol_index_by_date = prepared["symbol_index_by_date"]
    symbol_prev_pivot = prepared["symbol_prev_pivot"]
    symbol_prev_close = prepared["symbol_prev_close"]

    cash = INITIAL_CAPITAL
    open_positions = {}
    pending_entries = {}
    trades = []
    equity_curve = []
    skipped_invalid_stop = 0
    skipped_no_cash = 0
    skipped_position_limit = 0

    def find_candidates(day_data):
        return day_data[
            (day_data["hard_filter_status"] == "PASS")
            & (
                day_data["breakout_volume_percentile"]
                >= VOLUME_PCT_THRESHOLD
            )
        ]

    def try_open(symbol, entry_price, entry_date, signal_close, atr14, date):
        nonlocal cash, skipped_invalid_stop, skipped_no_cash
        nonlocal skipped_position_limit

        if symbol in open_positions:
            return
        if len(open_positions) >= MAX_CONCURRENT_POSITIONS:
            skipped_position_limit += 1
            return

        if stop_anchor == "trigger":
            stop_price = signal_close - ATR_STOP_MULTIPLIER * atr14
        elif stop_anchor == "filled":
            stop_price = entry_price - ATR_STOP_MULTIPLIER * atr14
        else:
            raise ValueError(f"Stop anchor tidak dikenal: {stop_anchor}")

        # Risiko riil harus dihitung dari harga beli aktual ke harga stop.
        risk_per_share = entry_price - stop_price
        if (
            pd.isna(stop_price)
            or pd.isna(risk_per_share)
            or stop_price <= 0
            or risk_per_share <= 0
        ):
            skipped_invalid_stop += 1
            return

        positions_value = sum(
            position["shares"]
            * symbol_row_by_date.get(open_symbol, {})
            .get(date, {})
            .get("close_raw", position["entry_price"])
            for open_symbol, position in open_positions.items()
        )
        current_equity = cash + positions_value
        risk_amount = current_equity * RISK_PCT
        shares = risk_amount / risk_per_share
        notional = shares * entry_price

        if notional > cash:
            skipped_no_cash += 1
            return

        cash -= notional
        open_positions[symbol] = {
            "shares": shares,
            "entry_price": entry_price,
            "trigger_price": signal_close,
            "atr14_t0": atr14,
            "stop_price": stop_price,
            "entry_date": entry_date,
        }

    for date in dates:
        day_data = by_date[date]

        # 0. Eksekusi sinyal sebelumnya pada open H+1.
        for symbol in list(pending_entries):
            signal = pending_entries.pop(symbol)
            if symbol not in day_data.index:
                continue

            today = day_data.loc[symbol]
            open_today = today.get("open_raw")
            if pd.isna(open_today) or open_today <= 0:
                continue

            try_open(
                symbol=symbol,
                entry_price=float(open_today),
                entry_date=date,
                signal_close=float(signal["signal_close"]),
                atr14=float(signal["atr14"]),
                date=date,
            )

        # 1. Exit, termasuk kemungkinan stop pada hari entry.
        for symbol in list(open_positions):
            if symbol not in day_data.index:
                continue

            row = day_data.loc[symbol]
            position = open_positions[symbol]
            entry_index = symbol_index_by_date[symbol].get(position["entry_date"])
            today_index = symbol_index_by_date[symbol].get(date)
            days_held = (
                today_index - entry_index
                if entry_index is not None and today_index is not None
                else 0
            )

            exit_reason = None
            exit_price = None
            if row["low_raw"] <= position["stop_price"]:
                exit_reason = "stop_loss"
                exit_price = position["stop_price"]
            elif days_held >= MAX_HOLDING_DAYS:
                exit_reason = "max_holding"
                exit_price = row["close_raw"]
            elif pd.notna(row.get("ema20")) and row["close_raw"] < row["ema20"]:
                exit_reason = "trend_exit"
                exit_price = row["close_raw"]

            if exit_reason is not None:
                proceeds = position["shares"] * exit_price
                pnl = proceeds - position["shares"] * position["entry_price"]
                cash += proceeds
                trades.append(
                    {
                        "symbol": symbol,
                        "entry_date": position["entry_date"],
                        "exit_date": date,
                        "trigger_price": position["trigger_price"],
                        "entry_price": position["entry_price"],
                        "atr14_t0": position["atr14_t0"],
                        "stop_price": position["stop_price"],
                        "exit_price": exit_price,
                        "shares": position["shares"],
                        "pnl": pnl,
                        "pnl_pct": (
                            exit_price / position["entry_price"] - 1
                        )
                        * 100,
                        "days_held": days_held,
                        "exit_reason": exit_reason,
                    }
                )
                del open_positions[symbol]

        # 2. Sinyal T0 disimpan untuk entry open H+1.
        for symbol, row in find_candidates(day_data).iterrows():
            if symbol in open_positions or symbol in pending_entries:
                continue

            previous_pivot = symbol_prev_pivot.get(symbol, {}).get(date)
            if (
                previous_pivot is None
                or pd.isna(previous_pivot)
                or not row["close_raw"] > previous_pivot
            ):
                continue
            if pd.isna(row.get("atr14")) or row["atr14"] <= 0:
                continue

            previous_close = symbol_prev_close.get(symbol, {}).get(date)
            if previous_close is None or pd.isna(previous_close) or previous_close <= 0:
                continue

            pending_entries[symbol] = {
                "signal_date": date,
                "signal_close": row["close_raw"],
                "atr14": row["atr14"],
            }

        # 3. Mark-to-market untuk equity curve dan drawdown.
        positions_value = 0.0
        for symbol, position in open_positions.items():
            price = symbol_row_by_date.get(symbol, {}).get(date, {}).get("close_raw")
            if price is None or pd.isna(price):
                price = position["entry_price"]
            positions_value += position["shares"] * price

        equity_curve.append(
            {
                "date": date,
                "cash": cash,
                "positions_value": positions_value,
                "n_open_positions": len(open_positions),
                "total_equity": cash + positions_value,
            }
        )

    diagnostics = {
        "n_open_at_end": len(open_positions),
        "n_pending_at_end": len(pending_entries),
        "n_skipped_invalid_stop": skipped_invalid_stop,
        "n_skipped_no_cash": skipped_no_cash,
        "n_skipped_position_limit": skipped_position_limit,
    }
    return pd.DataFrame(trades), pd.DataFrame(equity_curve), diagnostics


def profit_factor(trades):
    gross_win = trades.loc[trades["pnl"] > 0, "pnl"].sum()
    gross_loss = -trades.loc[trades["pnl"] < 0, "pnl"].sum()
    return gross_win / gross_loss if gross_loss > 0 else float("inf")


def profit_factor_ex_top10(trades):
    top_symbols = (
        trades.groupby("symbol")["pnl"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .index
    )
    return profit_factor(trades[~trades["symbol"].isin(top_symbols)])


def max_drawdown_pct(equity):
    values = equity["total_equity"].astype(float)
    running_peak = values.cummax()
    drawdown = values / running_peak - 1.0
    return drawdown.min() * 100


def summarize(label, trades, equity, diagnostics):
    final_equity = equity["total_equity"].iloc[-1]
    n_trades = len(trades)
    result = {
        "variant": label,
        "n_trades": n_trades,
        "win_rate_pct": (trades["pnl"] > 0).mean() * 100 if n_trades else np.nan,
        "profit_factor": profit_factor(trades) if n_trades else np.nan,
        "pf_ex_top10": profit_factor_ex_top10(trades) if n_trades else np.nan,
        "total_return_pct": (final_equity / INITIAL_CAPITAL - 1) * 100,
        "max_drawdown_pct": max_drawdown_pct(equity),
        "average_pnl_pct": trades["pnl_pct"].mean() if n_trades else np.nan,
        "median_pnl_pct": trades["pnl_pct"].median() if n_trades else np.nan,
        "average_days_held": trades["days_held"].mean() if n_trades else np.nan,
    }
    result.update(diagnostics)
    return result


print("Load feature store...")
feature_store = pd.read_parquet(FEATURE_STORE_PATH)
feature_store["date"] = pd.to_datetime(feature_store["date"])

required_columns = {
    "date",
    "symbol",
    "open_raw",
    "high_raw",
    "low_raw",
    "close_raw",
    "atr14",
    "ema20",
    "pivot_high",
    "breakout_volume_percentile",
}
missing_columns = sorted(required_columns - set(feature_store.columns))
if missing_columns:
    raise ValueError(f"Feature store kehilangan kolom: {missing_columns}")

print(
    f"Rows={len(feature_store):,} | Symbols={feature_store['symbol'].nunique():,} | "
    f"Dates={feature_store['date'].min().date()} s/d {feature_store['date'].max().date()}"
)

print("Compute hard filter dan lookup bersama...")
prepared_data = prepare_data(feature_store)

summaries = []
trade_logs = {}
equity_logs = {}

for variant_label, anchor in VARIANTS.items():
    print(f"Run {variant_label}...")
    variant_trades, variant_equity, variant_diagnostics = backtest(
        prepared_data, stop_anchor=anchor
    )
    summaries.append(
        summarize(
            variant_label,
            variant_trades,
            variant_equity,
            variant_diagnostics,
        )
    )
    trade_logs[variant_label] = variant_trades
    equity_logs[variant_label] = variant_equity

summary = pd.DataFrame(summaries)
numeric_columns = summary.select_dtypes(include="number").columns
summary[numeric_columns] = summary[numeric_columns].round(3)

print("\n=== A/B STOP ANCHOR ===")
print(summary.to_string(index=False))

summary_filename = "ussy_trendfoll_stop_anchor_ab_summary.csv"
summary.to_csv(summary_filename, index=False)
shutil.copy(summary_filename, f"{DRIVE_FOLDER}/{summary_filename}")

for variant_label in VARIANTS:
    trades_filename = f"ussy_trendfoll_stop_anchor_{variant_label}_trades.csv"
    equity_filename = f"ussy_trendfoll_stop_anchor_{variant_label}_equity.csv"
    trade_logs[variant_label].to_csv(trades_filename, index=False)
    equity_logs[variant_label].to_csv(equity_filename, index=False)
    shutil.copy(trades_filename, f"{DRIVE_FOLDER}/{trades_filename}")
    shutil.copy(equity_filename, f"{DRIVE_FOLDER}/{equity_filename}")

print(f"\nSemua output tersimpan di {DRIVE_FOLDER}")
print("Mengunduh ringkasan...")
files.download(summary_filename)
print("Selesai.")
